[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap32_aposentadoria_saques.ipynb)

# Capítulo 32: Aposentadoria II: previdência privada e a fase de usufruto

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Você testa taxas de saque, vê o efeito da ordem dos retornos (o risco de sequência) e simula milhares de futuros possíveis para a sua aposentadoria.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. Quanto a taxa de saque rende por mês

In [ ]:
PATRIMONIO = 1_000_000   # MUDE AQUI
for taxa in (0.03, 0.035, 0.04, 0.045):
    print(f'Taxa de {pct(taxa,1)}: {brl(PATRIMONIO*taxa)} no primeiro ano, ou {brl(PATRIMONIO*taxa/12)} por mês')

## 2. O risco de sequência
O exemplo do capítulo 32: dois aposentados, os mesmos retornos em ordem diferente, saque de R$ 60 mil no início de cada ano.

In [ ]:
def trajetoria(inicial, saque, retornos):
    v = inicial
    for r in retornos:
        v = (v - saque) * (1 + r)
    return v
ana = trajetoria(1_000_000, 60_000, [-0.25, 0.15, 0.15])
beto = trajetoria(1_000_000, 60_000, [0.15, 0.15, -0.25])
print(f'Ana (queda no primeiro ano): {brl(ana)}')
print(f'Beto (queda no último ano): {brl(beto)}')
print(f'Diferença: {brl(beto - ana)}, só por causa da ordem')

## 3. Milhares de futuros possíveis
Simulação com retornos aleatórios. Média e volatilidade são **hipóteses**, em termos reais (acima da inflação). Não é previsão: é um teste de resistência.

In [ ]:
SAQUE_ANUAL = 40_000      # MUDE AQUI: em reais de hoje
ANOS = 30                 # MUDE AQUI
MEDIA_REAL = 0.04         # MUDE AQUI: retorno médio real ao ano
VOL = 0.10                # MUDE AQUI: volatilidade anual
N = 10_000
rng = np.random.default_rng(42)
ret = rng.normal(MEDIA_REAL, VOL, size=(N, ANOS))
v = np.full(N, float(PATRIMONIO)); vivo = np.ones(N, bool); caminhos = []
for a in range(ANOS):
    v = np.where(vivo, (v - SAQUE_ANUAL) * (1 + ret[:, a]), 0)
    vivo &= v > 0; v = np.maximum(v, 0); caminhos.append(v.copy())
cam = np.array(caminhos)
print(f'Em {pct(vivo.mean(),1)} dos cenários o dinheiro durou {ANOS} anos.')
fig, ax = plt.subplots()
for q, cor in ((10, CORES['vermelho']), (50, CORES['azul']), (90, CORES['verde'])):
    ax.plot(range(1, ANOS+1), np.percentile(cam, q, axis=1), color=cor, lw=2, label=f'{q}% dos cenários abaixo desta linha')
ax.set_xlabel('Anos de aposentadoria'); ax.set_title(f'Patrimônio com saque de {brl(SAQUE_ANUAL,0)} por ano'); eixo_reais(ax); ax.legend(); plt.show()

## 4. O peso da taxa de administração
O exemplo do capítulo 32: R$ 500 por mês por 25 anos, a 4% ou a 3% ao ano acima da inflação.

In [ ]:
def fv(aporte, taxa_ano, anos):
    i = (1 + taxa_ano) ** (1/12) - 1
    return aporte * (((1 + i) ** (anos*12) - 1) / i)
print('A 4% ao ano:', brl(fv(500, 0.04, 25)))
print('A 3% ao ano:', brl(fv(500, 0.03, 25)))

Teste o seu número na [calculadora de aposentadoria](https://www.traderbrasil.com/blog/calculadora-de-aposentadoria-quanto-preciso-acumular.php) e compare os regimes na [calculadora de PGBL e VGBL](https://www.traderbrasil.com/blog/calculadora-pgbl-vgbl-progressiva-regressiva.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 32 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap32/

Achou um erro? Escreva para contato@traderbrasil.com.